In [1]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import AutoTokenizer, EsmForMaskedLM, PretrainedConfig

from transformers import AutoModelForCausalLM
from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops
import yaml
import sys
import json
import functools

import numpy as np

/home/averma2/miniforge3/envs/exploratory_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences
from interp_utils import get_hooked_state_dict, get_hooked_esm_config, get_logits_hooked_esm

In [3]:
device = "cuda"

In [4]:
# ESM-2 config
with open("./esm_config150M.json", "r") as file:
    esm_config = json.load(file)
model_name = esm_config["_name_or_path"]
model_name = model_name[model_name.find("facebook"):]
esm_config["token_dropout"] = False
esm_config["model_name"] = model_name
esm_config = PretrainedConfig(**esm_config)

# hooked ESM-2 config
hooked_esm_config = get_hooked_esm_config(esm_config)

In [5]:
# ESM-2 35M
model_name = esm_config.model_name
tokenizer = AutoTokenizer.from_pretrained(model_name)
reg_esm = EsmForMaskedLM(esm_config).to(device)

# hooked ESM-2
hooked_esm = HookedTransformer(hooked_esm_config)
hooked_esm.load_state_dict(get_hooked_state_dict(reg_esm.state_dict(), hooked_esm_config))

# helper function to get logits from hooked ESM-2 head
apply_esm_lm_head = functools.partial(get_logits_hooked_esm, ESM2_lm_head=reg_esm.lm_head)
print(model_name)

facebook/esm2_t30_150M_UR50D


<img src="./ESM2_architecture.png" width="1000"/>

In [6]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
    
pathogens = list(config["pathogens"].keys())
pathogen_idx = 0
print(pathogens[pathogen_idx])
fasta_file = f"../../data/pathogen/{pathogens[pathogen_idx]}/alignment.fasta"
data = load_sequences(fasta_file)
sequence_names, sequences = list(zip(*list(data.items())))

influenza_h3n2_ha


In [7]:
N = 100
test_seq = sequences[N]
tokenized_seq = tokenizer(test_seq, return_tensors="pt").input_ids[:,:hooked_esm_config.n_ctx].to(device)

In [8]:
# run respective models
with torch.no_grad():
    outputs = reg_esm.forward(tokenized_seq, output_hidden_states=True)
    reg_esm_hs = outputs.hidden_states

hooked_toks, cache = hooked_esm.run_with_cache(tokenized_seq)

In [9]:
for l in range(esm_config.num_hidden_layers + 2):
    # 0th layer is embedding layer
    if l < esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"blocks.{l}.hook_resid_pre"]
    
    # final layer
    elif l == esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"ln_final.hook_normalized"]

    # ESM LM head:
    else:
        actual_output = outputs.logits
        with torch.no_grad():
            hooked_output = apply_esm_lm_head(hooked_toks)
    
    is_close = torch.all(torch.isclose(actual_output, hooked_output, rtol=0.1)).item()
    print(f"layer {l}: {is_close}")

layer 0: True
layer 1: True
layer 2: True
layer 3: True
layer 4: False
layer 5: True
layer 6: True
layer 7: True
layer 8: False
layer 9: True
layer 10: False
layer 11: True
layer 12: True
layer 13: True
layer 14: True
layer 15: True
layer 16: True
layer 17: True
layer 18: False
layer 19: True
layer 20: True
layer 21: False
layer 22: True
layer 23: True
layer 24: False
layer 25: True
layer 26: False
layer 27: True
layer 28: True
layer 29: True
layer 30: True
layer 31: True
